In [1]:
import os
import numpy as np
import tifffile
import sqlite3
import argparse
from datetime import datetime
from cellpose import models

import sys
sys.path.insert(0, '/home/boyesh/akoya_pcf/scripts')
sys.argv = ['segmentation.py', '--project', 'prototype']

from utils import is_already_processed
from ingestion import scan_for_files

tile_size = 2048
tile_overlap = 256

ISILON_BASE = os.environ.get("AKOYA_ISILON")
DB_PATH = os.environ.get("AKOYA_DB")

In [ ]:
from segmentation import extract_dapi_channel, tile_image, detect_cells, save_masks

In [3]:
file = "/home/boyesh/IMLAkoyafusion/prototype/052024 P7HuP120 #03 SG03_Scan1.unmixed.qptiff"

dapi = extract_dapi_channel(file)
print(f"DAPI shape: {dapi.shape}")

DAPI shape: (37440, 30720)


In [4]:
tiles = tile_image(dapi)
print(f"Number of tiles: {len(tiles)}")
print(f"First tile array shape: {tiles[0]['array'].shape}")
print(f"Last tile array shape: {tiles[-1]['array'].shape}")

Number of tiles: 378
First tile array shape: (2048, 2048)
Last tile array shape: (1600, 256)


In [5]:
model = models.CellposeModel(gpu=False)
masks, flows, styles = model.eval(tiles[-1]['array'], diameter=None)
print(f"mask shape: {masks.shape}")

mask shape: (1600, 256)


In [6]:
print(f"Last tile row_start: {tiles[-1]['row_start']}")
print(f"Last tile row_stop: {tiles[-1]['row_stop']}")
print(f"Last tile col_start: {tiles[-1]['col_start']}")
print(f"Last tile col_stop: {tiles[-1]['col_stop']}")
print(f"output size would be: {tiles[-1]['row_stop']} x {tiles[-1]['col_stop']}")

Last tile row_start: 35840
Last tile row_stop: 37440
Last tile col_start: 30464
Last tile col_stop: 30720
output size would be: 37440 x 30720


In [7]:
tiles = tile_image(dapi)
print(f"Last tile row_stop: {tiles[-1]['row_stop']}")
print(f"Last tile col_stop: {tiles[-1]['col_stop']}")

Last tile row_stop: 37440
Last tile col_stop: 30720


In [8]:
print(f"Output would be: {tiles[-1]['row_stop']} x {tiles[-1]['col_stop']}")

Output would be: 37440 x 30720


In [10]:
def save_masks(masks, file_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output dir exists: {os.path.exists(output_dir)}")

In [3]:
import os
import sqlite3

conn = sqlite3.connect("/home/boyesh/akoya_pcf/akoya.db")
cursor = conn.cursor()
cursor.execute("DELETE FROM segmentation_results")
conn.commit()
conn.close()
print("Cleared segmentation_results")

Cleared segmentation_results


In [2]:
import sqlite3

import os
os.environ["AKOYA_DB"] = "/home/boyesh/akoya_pcf/akoya.db"
DB_PATH = os.environ.get("AKOYA_DB")

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM segmentation_results")
print(cursor.fetchone())

cursor.execute("SELECT * FROM pipeline_status")
print(cursor.fetchall())


(1,)
[(1, 1, 'Not Started', 'Failed', 'Passed', 'Not Started', 'Not Started', 'Not Started', None, None), (2, 2, 'Not Started', 'Failed', 'Passed', 'Not Started', 'Not Started', 'Not Started', None, None), (3, 3, 'Not Started', 'Complete', 'Not Started', 'Not Started', 'Not Started', 'Not Started', None, None), (4, 4, 'Not Started', 'Complete', 'Not Started', 'Not Started', 'Not Started', 'Not Started', None, None), (5, 5, 'Not Started', 'Failed', 'Not Started', 'Not Started', 'Not Started', 'Not Started', None, None)]


In [3]:
cursor.execute("SELECT slide_id, cell_count, segment_status FROM segmentation_results")
print(cursor.fetchall())

[(1, 193758, 'Passed')]
